# **Initialization**

In [1]:
print('Start')

Start


In [2]:
import glob
import random
import os
import time as pytime
import pandas as pd
import modified_didppy as m_dp

# **Data**

In [3]:
def read_tsp_cappart_format(file_path):
    """
    Parses the TSP/TSPTW text files from the specified directory.
    Structure:
    - n (int)
    - n*n distance matrix entries
    - n*2 time window entries (ignored for TSP)
    - n x_coords (ignored)
    - n y_coords (ignored)
    """
    with open(file_path, 'r') as f:
        # split() handles all whitespace (newlines and spaces) automatically
        values = f.read().split()

    iterator = iter(values)
    
    try:
        # 1. Read Number of Nodes
        n = int(next(iterator))
        
        # 2. Read Distance Matrix (n x n)
        # The file contains a flattened list of integer distances
        c = []
        for i in range(n):
            row = []
            for j in range(n):
                val = float(next(iterator)) # Read as float first to be safe
                row.append(float(val))        # Convert to int as per your DIDP model type
            c.append(row)
            
        # The rest of the file (Time windows, coords) is ignored for pure TSP
        # but the iterator ensures we consumed exactly what we needed.
        num_locations = n
        travel_cost = c
        return num_locations, travel_cost

    except StopIteration:
        raise ValueError(f"File {file_path} ended unexpectedly.")

# **Create DIDP model**

In [4]:
def creation_of_didp_model_function():
# --- A. Read Data ---
    # Using the function you defined in previous cells
    num_locations, travel_cost = current_num_locations, current_travel_cost
    
    # --- B. Initialize Model (MUST be done fresh for every instance) ---
    model = m_dp.Model(maximize=False, float_cost=True)
    customer = model.add_object_type(number=num_locations)
    
    # State Variables
    unvisited = model.add_set_var(object_type=customer, target=list(range(1, num_locations)))
    location = model.add_element_var(object_type=customer, target=0)
    
    # Resource Tables
    travel_time = model.add_float_table(travel_cost)
    
    # Transitions: Visit customer j
    for j in range(1, num_locations):
        visit = m_dp.Transition(
            name="visit {}".format(j),
            cost=travel_time[location, j] + m_dp.FloatExpr.state_cost(),
            preconditions=[unvisited.contains(j)],
            effects=[
                (unvisited, unvisited.remove(j)),
                (location, j),
            ],
        )
        model.add_transition(visit)
    
    # Transitions: Return to depot
    return_to_depot = m_dp.Transition(
        name="return",
        cost=travel_time[location, 0] + m_dp.FloatExpr.state_cost(),
        effects=[(location, 0)],
        preconditions=[unvisited.is_empty(), location != 0],
    )
    model.add_transition(return_to_depot)
    
    # Base Case
    model.add_base_case([unvisited.is_empty(), location == 0])
    
    # --- C. Dual Bounds ---
    # Min outgoing edge
    min_to = model.add_float_table(
        [min(travel_cost[k][j] for k in range(num_locations) if k != j) for j in range(num_locations)]
    )
    model.add_dual_bound(min_to[unvisited] + (location != 0).if_then_else(min_to[0], 0))
    
    # Min incoming edge
    min_from = model.add_float_table(
        [min(travel_cost[j][k] for k in range(num_locations) if k != j) for j in range(num_locations)]
    )
    model.add_dual_bound(
        min_from[unvisited] + (location != 0).if_then_else(min_from[location], 0)
    )

    
    didp_bundle = model
    return didp_bundle

In [16]:
# ==========================================
# 1. Configuration & File Selection
# ==========================================

# Directory containing the instances (Update this path if needed)
folder_path = r"C:\Users\ACER\Desktop\Code\0.Thesis implementation\2_DIDP_custom_search_guidance_local\Thesis_modified_DIDP\1_TSP_dual_bounds_and_models\Datasets\n50"

# Get all .txt files
all_files = glob.glob(os.path.join(folder_path, "*.txt"))

# Select 20 random instances (or all if less than 20)
num_instances_to_test = 20
if len(all_files) > num_instances_to_test:
    selected_files = random.sample(all_files, num_instances_to_test)
else:
    selected_files = all_files

print(f"Found {len(all_files)} files. Selected {len(selected_files)} for testing.")
print("Selected Instances:")
for f in selected_files:
    print(f" - {os.path.basename(f)}")
print("-" * 50)



Found 100 files. Selected 20 for testing.
Selected Instances:
 - 61.txt
 - 53.txt
 - 50.txt
 - 89.txt
 - 31.txt
 - 75.txt
 - 52.txt
 - 33.txt
 - 39.txt
 - 11.txt
 - 14.txt
 - 54.txt
 - 78.txt
 - 59.txt
 - 46.txt
 - 4.txt
 - 79.txt
 - 48.txt
 - 22.txt
 - 87.txt
--------------------------------------------------


In [17]:
# ==========================================
# 2. Testing Loop
# ==========================================

results_data = []
output_csv_name = "TSP_single_dual_bound_50_cus_results.csv"

# Ensure global variables exist (initializing them)
current_num_locations = 0
current_travel_cost = []

for i, file_path in enumerate(selected_files):
    instance_name = os.path.basename(file_path)
    print(f"\n[{i+1}/{len(selected_files)}] Processing: {instance_name}")
    
    try:
        # --- A. Read Data & Update Globals ---
        # We read data and IMMEDIATELY update the global variables that 
        # creation_of_didp_model_function relies on.
        n_loc, t_cost = read_tsp_cappart_format(file_path)
        
        # Inject into globals
        current_num_locations = n_loc
        current_travel_cost = t_cost

        # --- B. Initialize Model ---
        # We call your existing function, which reads the globals we just set
        didp_bundle = creation_of_didp_model_function()
        model= didp_bundle # Unpack the tuple
        
        # --- C. Solver Execution ---
        t_start = pytime.time()
        
        # Solver with 30 minute limit (1800 seconds) or similar to TSP config
        solver = m_dp.CABS(
            model,
            quiet=False, # Set to True to reduce console spam
            time_limit=10
        )
        
        solution = solver.search()
        
        t_end = pytime.time()
        duration = t_end - t_start
        
        # --- D. Logging Results ---
        if solution.is_optimal:
             cost = solution.cost
             status = "True"
        elif solution.cost is not None:
             cost = solution.cost
             status = "False (Time Limit)"
        else:
             cost = "Inf"
             status = "False (No Sol)"

        nodes_gen = solution.generated
        nodes_exp = solution.expanded
        
        print(f"   -> Done. Cost: {cost}, Time: {duration:.2f}s, Optimal: {status}")

        # Append to results list
        results_data.append({
            "Instance": instance_name,
            "Cost": cost,
            "Nodes Expanded": nodes_exp,
            "Nodes Generated": nodes_gen,
            "Running Time (s)": duration,
            "Is Optimal": status
        })

    except Exception as e:
        print(f"   -> ERROR processing {instance_name}: {e}")
        # Optional: print full traceback if debugging
        # import traceback
        # traceback.print_exc()
        
        results_data.append({
            "Instance": instance_name,
            "Cost": "Error",
            "Nodes Expanded": 0,
            "Nodes Generated": 0,
            "Running Time (s)": 0,
            "Is Optimal": "Error"
        })

    # --- E. Intermediate Save ---
    # Save after every iteration for safety
    df_results = pd.DataFrame(results_data)
    df_results.to_csv(output_csv_name, index=False)

print("\n" + "="*50)
print("Batch Testing Complete.")
print(f"Results saved to {output_csv_name}")
print(df_results)


[1/20] Processing: 61.txt
   -> Done. Cost: 665.0, Time: 10.01s, Optimal: False (Time Limit)

[2/20] Processing: 53.txt
   -> Done. Cost: 653.0, Time: 10.02s, Optimal: False (Time Limit)

[3/20] Processing: 50.txt
   -> Done. Cost: 671.0, Time: 10.01s, Optimal: False (Time Limit)

[4/20] Processing: 89.txt
   -> Done. Cost: 618.0, Time: 10.02s, Optimal: False (Time Limit)

[5/20] Processing: 31.txt
   -> Done. Cost: 570.0, Time: 10.00s, Optimal: False (Time Limit)

[6/20] Processing: 75.txt
   -> Done. Cost: 612.0, Time: 10.01s, Optimal: False (Time Limit)

[7/20] Processing: 52.txt
   -> Done. Cost: 643.0, Time: 10.02s, Optimal: False (Time Limit)

[8/20] Processing: 33.txt
   -> Done. Cost: 679.0, Time: 10.01s, Optimal: False (Time Limit)

[9/20] Processing: 39.txt
   -> Done. Cost: 630.0, Time: 10.01s, Optimal: False (Time Limit)

[10/20] Processing: 11.txt
   -> Done. Cost: 662.0, Time: 10.00s, Optimal: False (Time Limit)

[11/20] Processing: 14.txt
   -> Done. Cost: 661.0, Time: 